# WAAM 로봇 3대 경로 계획 — 코랩에서 풀고 계산 시간 재기

INE4107 IC-PBL. 공개 형상 6개(또는 원하는 시나리오 폴더)를 우리 알고리즘(`src/solve.py`)으로 풀고,
형상마다 **계산 시간**을 잰 뒤 교수님 채점기(WAAM Validator)로 채점합니다.

- **런타임:** 무료 일반 CPU (메뉴 *런타임 → 런타임 유형 변경 → CPU*). GPU·TPU 는 쓰지 않습니다.
- **계산 시간:** 형상 폴더를 넣은 순간부터 `trajectory.csv` 가 나올 때까지(형상 하나당). 설치·채점 시간은 넣지 않습니다.
  미리 학습하는 단계는 없습니다.
- **확정적:** 무작위를 쓰지 않아 같은 환경에서는 몇 번을 돌려도 같은 궤적이 나옵니다.
- 위에서부터 차례로 실행하고, 맨 아래 표와 맨 위 컴퓨터 정보를 캡처하면 됩니다.

## 1. 컴퓨터 정보

In [ ]:
!echo "CPU 개수: $(nproc)"
!lscpu | grep -E "Model name|^CPU\(s\)"
!free -g | head -2
!python --version

## 2. 설치 (계산 시간에 넣지 않음)
우리 저장소와 채점기를 내려받고 채점기를 설치합니다. 1–2분 걸립니다.

In [ ]:
%%bash
cd /content
[ -d waam-icpbl ] || git clone -q https://github.com/hosukim03/waam-icpbl.git
cd waam-icpbl && git pull -q
[ -d WAAM_Validator ] || git clone -q https://github.com/Ha-An/WAAM_Validator.git
pip install -q -e WAAM_Validator
git log -1 --format='우리 코드: %h (%ci)'
git -C WAAM_Validator log -1 --format='채점기: %h (%ci)'

## 3. 풀기 — 형상마다 계산 시간
`SCENARIOS` 에 시나리오 폴더(`config.yaml`·`target.stl` 이 있는 폴더)를 넣습니다. 기본은 공개 6개입니다.
다른 형상을 풀려면 폴더를 올리고(왼쪽 파일 창) 그 경로를 넣으면 됩니다.

In [ ]:
import subprocess, sys, time
from pathlib import Path

ROOT = Path('/content/waam-icpbl')
OUT = Path('/content/out')
SCENARIOS = [ROOT / 'WAAM_Validator' / 'tests' / m for m in ['01', '02', '03', '04', '05', '06']]
LIMIT_S = 300.0      # 5분

times = {}
for scen in SCENARIOS:
    out = OUT / scen.name
    t0 = time.perf_counter()
    p = subprocess.run([sys.executable, str(ROOT / 'src' / 'solve.py'), str(scen), str(out)],
                       capture_output=True, text=True)
    times[scen.name] = time.perf_counter() - t0
    last = p.stdout.strip().splitlines()[-1] if p.returncode == 0 and p.stdout.strip() else p.stderr[-400:]
    print(f"{scen.name}: {times[scen.name]:6.1f} s  {'✅' if times[scen.name] <= LIMIT_S else '⛔ 5분 넘음'}  | {last}")

## 4. 채점 (계산 시간에 넣지 않음)

In [ ]:
import json
import pandas as pd

rows = []
for scen in SCENARIOS:
    out = OUT / scen.name
    v = subprocess.run([sys.executable, '-m', 'waam_validator.cli', 'run', str(out),
                        '--output', str(OUT / '_val' / scen.name), '--json'], capture_output=True, text=True)
    try:
        s = json.loads(v.stdout)
        rows.append({'형상': scen.name, '계산 시간(s)': round(times[scen.name], 1),
                     '5분 안': '✅' if times[scen.name] <= LIMIT_S else '⛔',
                     'makespan(s)': round(float(s['schedule']['makespan_s']), 1), '판정': s['status'],
                     'IoU(%)': round(100 * s['shape']['iou'], 2), '충돌': s['collision']['collision_event_count']})
    except Exception:
        rows.append({'형상': scen.name, '계산 시간(s)': round(times[scen.name], 1), '판정': '채점 오류',
                     '메시지': (v.stderr or v.stdout)[-200:]})
df = pd.DataFrame(rows)
total = {'형상': '합계', '계산 시간(s)': round(df['계산 시간(s)'].sum(), 1)}
if 'makespan(s)' in df:
    total['makespan(s)'] = round(df['makespan(s)'].sum(), 1)
    total['판정'] = f"{(df['판정'] == 'PASS').sum()}/{len(df)} PASS"
pd.concat([df, pd.DataFrame([total])], ignore_index=True)

## 5. 캡처
위 표(형상별 계산 시간·makespan·판정)와 1번의 컴퓨터 정보를 함께 캡처합니다.
형상마다 고른 설정과 단계별 시간(경로 계산, 계획, 국소 탐색)은 `/content/out/<형상>/solve.txt` 에 있습니다.